In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
import seaborn as sns
import sys
sys.path.append('../../scripts')
import json_parser # type: ignore

pd.set_option('display.max_columns', None)

In [ ]:
json_file = '../../data/raw/AN2023.json' 

df = pd.read_json(json_file)
df = json_parser.flatten_data(json_file)

print(df.shape) # Rows, Columns


In [ ]:
df.head()

## Missing values

In [ ]:
# features with missing values
features_na = [features for features in df.columns if df[features].isnull().sum() > 1]

# feature name + percentage of missing values
for feature in features_na:
    print(feature, np.round(df[feature].isnull().mean(), 5), '% missing values')

### Relationship between missing values and the output (EPnren)

In [ ]:

for feature in features_na:
    data = df.copy()
    
    # 1 to indicate missing, 0 NOT missing
    data[feature] = np.where(data[feature].isnull(), 1, 0)
    
    # calculate the mean of the target for both groups
    data.groupby(feature)['epglnren'].median().plot.bar()
    plt.title(feature)
    plt.show()

## Numerical values

In [ ]:
numerical_features = df.select_dtypes(include=['number']).columns.tolist()
print('Number of numerical features: ', len(numerical_features))

df[numerical_features].head()

## Temporal variables

In [ ]:
time_features = [feature for feature in df if 'sopralluogo' in feature or 'validita' in feature]
time_features

In [ ]:
# exploring the content of these features

for feature in time_features:
    print(feature, df[feature].unique())  # did not consider anno_costruzione because it is a numerical predictor, not a time feature

### Is there any relation between anno_costruzione and epglnren?

In [ ]:
df.groupby('anno_costruzione')['epglnren'].median().plot.bar(figsize=(12, 6))
plt.xlabel('anno_costruzione')
plt.ylabel('Median epglnren')
plt.title('Median epglnren by anno_costruzione')
plt.xticks(rotation=70, ha='right', fontsize=5)
plt.tight_layout()
plt.show()

##### no further data exploration required for the temporal variables, since they have no relation to the prediction output

## Continuous and Discrete variables

In [ ]:
discrete_features = [feature for feature in df if len(df[feature].unique()) < 5 and feature not in time_features]
print('Discrete features count: ', len(discrete_features))

In [ ]:
discrete_features

In [ ]:
# exploring the relationship between the discrete features and the target variable epglnren
for feature in discrete_features:
    df.groupby(feature)['epglnren'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('egplnren')
    plt.title(feature)
    plt.show()

In [ ]:
# unstructured mixed text features (most probably unwanted and not useful for the prediction task)
unstructured_text_features = ['software_utilizzato', 'cap', 'comune', 'codice_istat_comune', 'informazioni_aggiuntive', 'informazioni_miglioramento', 'piano', 'altra_motivazione', 'provincia', 'comune']
print('Unstructured text features count: ', len(unstructured_text_features))

print('Unstructured text features: ', unstructured_text_features)

In [ ]:
# Continuous features
continuous_features = [feature for feature in df if feature not in discrete_features + time_features + unstructured_text_features and feature != 'classe_energetica' and feature != 'classe_energetica_raggiungibile']
print('Continuous features count: ', len(continuous_features))

In [ ]:
# histograms for continuous features
for feature in continuous_features:
    data = df.copy()
    data[feature].hist(bins=30)
    plt.xlabel(feature)
    plt.ylabel('Count')
    plt.title(feature)
    plt.show()

In [ ]:
## logarithmic transformation for skewed features
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    
    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data['epglnren'] = np.log(data['epglnren'])
        plt.scatter(data[feature], data['epglnren'])
        plt.xlabel(feature)
        plt.ylabel('epglnren')
        plt.title(feature)
        plt.show()
    else:
        print(f'{feature} has no positive values, skipping.')

## Outliers

In [ ]:
for feature in continuous_features:
    data = df.copy()
    data = data[(data[feature] > 0) & (data['epglnren'] > 0)]
    
    if len(data) > 0:
        data[feature] = np.log(data[feature])
        data.boxplot(column=feature)
        plt.ylabel(feature)
        plt.title(feature)
        plt.show()
    else:
        print(f'{feature} has no positive values, skipping.')

In [ ]:
## Categorical features
categorical_features = ['classe_energetica', 'classe_energetica_raggiungibile', 'zona_climatica']
categorical_features

In [ ]:
df[categorical_features].head()

In [ ]:
for feature in categorical_features:
    print(f'feature: {feature}, number of categories: {len(df[feature].unique())}')

In [ ]:
## relationship between categorical features and the target variable epglnren
for feature in categorical_features:
    df.groupby(feature)['epglnren'].median().plot.bar()
    plt.xlabel(feature)
    plt.ylabel('epglnren')
    plt.title(feature)
    plt.show()